> Processing of the results from training (including *clusterization*)

In [1]:
import pathlib

import yaml

import pandas as pd
import numpy as np

import jax.numpy as jnp
import jax.random as jr
from numpyro import distributions as dist

from torch.utils.data import DataLoader

import trapallada.data
import trapallada.analysis
import trapallada.results.analysis

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# load_dir

The base directory

In [3]:
base_dir = trapallada.util.get_base_directory()
base_dir

PosixPath('/home/manu/Sync/git/trapallada')

In [4]:
# results_dir = base_dir / 'results' / 'multichannel' / 'rates_large_infs' / 'fft_z_30'
results_dir = base_dir / 'results' / 'beta' / 'rates_large_infs' / 'grid2' / 'beta_0.001_dim_z_10/'
assert results_dir.exists()

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [5]:
ds, positions_df, model, training_df, parameters = trapallada.results.analysis.load_dir(results_dir)

For t-SNE

In [6]:
n_projected_components = 2

A `DataLoader` for the `Dataset`

In [7]:
dl = DataLoader(ds, batch_size=128, shuffle=False)

# project_onto_latent_space

Projection onto the latent space...

In [8]:
z = trapallada.results.analysis.project_onto_latent_space(dl, model.encoder)
z.shape

(30856, 10)

In [9]:
z_mean, z_sd = trapallada.results.analysis.project_onto_latent_space(dl, model.encoder, return_sd=True)
z_mean.shape, z_sd.shape

((30856, 10), (30856, 10))

In [10]:
type(z_mean), type(z_sd)

(jaxlib.xla_extension.ArrayImpl, jaxlib.xla_extension.ArrayImpl)

In [11]:
assert jnp.allclose(z, z_mean).item()

# wasserstein

In [12]:
d = trapallada.results.analysis.wasserstein_between_gaussians(z_mean, z_sd)
d.shape

(30856, 30856)

One might think of using JAX...

In [13]:
d_first = trapallada.results.analysis.wasserstein_between_one_and_many_gaussians(z_mean[0], z_sd[0], z_mean, z_sd)
assert (d_first == d[0]).all().item()
d_first

Array([0.        , 2.3509877 , 0.71233296, ..., 3.2207985 , 4.2187386 ,
       3.0590036 ], dtype=float32)

...but it yields a memory error (and *chunking* doesn't seem to yield a boost)

In [14]:
# jax.vmap(wasserstein_between_one_and_many_gaussians, in_axes=(0, 0, None, None))(z_mean, z_sd, z_mean, z_sd)

In [15]:
del d

In [16]:
clusters = trapallada.results.analysis.wasserstein_cluster(z_mean, z_sd, n_clusters=4)
clusters.shape, np.unique(clusters)

((30856,), array([1, 2, 3, 4], dtype=int32))

# tsne_project

...which is, in turn, projected, for visualization, using t-SNE

In [17]:
tsne = trapallada.results.analysis.tsne_project(z, n_projected_components=n_projected_components)

A `DataFrame` with metadata

In [18]:
res_df = ds.metadata_df.copy()

Names of columns to be added to the `DataFrame` for the latent space and t-SNE

In [19]:
z_cols = [f'z_{i}' for i in range(1, z.shape[-1]+1)]
tsne_cols = [f'tsne_{i}' for i in range(1, n_projected_components+1)]

# hdbscan_cluster

Clustering using [HDBSCAN](https://github.com/scikit-learn-contrib/hdbscan) with pretty much every default parameter. See [this notebook](https://nbviewer.org/github/scikit-learn-contrib/hdbscan/blob/master/notebooks/How%20HDBSCAN%20Works.ipynb) for an explanation.

In [20]:
res_df['cluster in z'] = trapallada.results.analysis.hdbscan_cluster(z, 10)
res_df['cluster in projected space'] = trapallada.results.analysis.hdbscan_cluster(tsne, 80)

/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/manu/Sync/git/trapallada/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


# augment_with_positions

Position information extracted from the *headers* of the FITs files is merged with the above `DataFrame`

In [21]:
res_df = trapallada.results.analysis.augment_with_positions(res_df, positions_df)

# augment_with_simbad

Some *alleged* labels associated with spatial coordinates from SIMBAD (XMATCH) database are appended

In [22]:
catalog_file = base_dir / 'experiments' / 'Cross_Simbad_r3.5.csv'
    
res_df = trapallada.results.analysis.augment_with_simbad(res_df, catalog_file)

/home/manu/Sync/git/trapallada/trapallada/results/analysis.py:125: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  catalog_df = pd.read_csv(catalog_file)


# augment_with_window_and_signal_ids

Two new columns are added
- one just combines the *tar* and *FIT* file, which univocally determines the signal,
- the other yields the number of window *within* every (*tar* file, *FIT* file) combination

In [23]:
res_df = trapallada.results.analysis.augment_with_window_and_signal_ids(res_df)

# i_windows_of_interest

In [29]:
for e in trapallada.results.analysis.i_windows_of_interest(res_df):
    display(e)

(Index([12284, 12288, 12292, 12296, 12300, 12304, 12308, 12312, 12316, 12320,
        12324, 12328, 12332, 12336, 12340, 12344, 12348, 12352, 12356, 12360,
        12364, 12368, 12372, 12376, 12380, 12384, 12388, 12392, 12396],
       dtype='int64'),
 'P0605950101PNX000SRCTSR8001 (Javi)')

(Index([26781, 26782, 26783, 26784, 26785, 26786, 26787, 26788, 26789, 26790,
        26791, 26792, 26793, 26794, 26795, 26796, 26797, 26798, 26799, 26800,
        26801, 26802, 26803, 26804, 26805],
       dtype='int64'),
 'P0690744401PNX000SRCTSR8002 (I)')

(Index([8222, 8224, 8226, 8228, 8230, 8232, 8234, 8236, 8238, 8240, 8242, 8244,
        8246, 8248],
       dtype='int64'),
 'P0740290301PNX000SRCTSR8002 (III)')

(Index([16827, 16828, 16829, 16830, 16831, 16832, 16833, 16834, 16835, 16836,
        16837, 16838, 16839, 16840],
       dtype='int64'),
 'P0720173901PNX000SRCTSR8001 (IV)')

In [31]:
type(e[0])

pandas.core.indexes.base.Index

# mahalanobis_based_metrics

In [25]:
n_labels = 100
y = dist.Bernoulli(0.4).sample(jr.PRNGKey(42), (n_labels,))
i_windows_of_interest = jr.choice(jr.PRNGKey(42), n_labels, (n_labels,)).tolist()

In [26]:
trapallada.results.analysis.mahalanobis_based_metrics(z_mean, y, i_windows_of_interest=i_windows_of_interest)

{'roc_auc': 0.4715073529411765,
 'average_precision': 0.2973973340181696,
 'best_recall_at_min_precision': 0.0}